# DBZD Phase 0 Kaggle runner
Attach all prior run archives for Batch 1; the runner merges every phase0_final_r3 run it finds under /kaggle/input. Download dbzd-runs.zip and attach that consolidated output for Batch 2. No tokens are required for the public repository.

In [ ]:
REPO_URL = "https://github.com/Alessio2405/DBZD.git"
RUN_BATCHES = {
    1: [("dbzd_full", 44), ("dbzd_stopgrad", 42)],
    2: [("dbzd_stopgrad", 43), ("dbzd_stopgrad", 44)],
}
BATCH_ID = 1  # Run Version 1, then 2; attach all prior result archives.
RUN_MATRIX = RUN_BATCHES[BATCH_ID]
CONFIG = "configs/default.yaml"
EXPERIMENT_REVISION = "phase0_final_r3"
DATAGEN_SCHEMA_VERSION = 3
RUNS_INPUT_DIR = ""  # Optional explicit directory containing prior runs.
RUNS_INPUT_ZIP = ""  # Optional legacy single archive path.
RUNS_INPUT_ZIPS = []  # Optional list of prior runs.zip/dbzd-runs.zip paths.
AUTO_DISCOVER_RUN_INPUTS = True  # Merge all matching /kaggle/input sessions.
DATASET_TITLE = "DBZD Phase 0 runs"
RUN_PROBES = True
PUBLISH_AFTER_EACH_RUN = False
AUTO_FIX_P100 = True


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

work = Path("/kaggle/working")
repo = work / "dbzd"
if repo.exists():
    shutil.rmtree(repo)

subprocess.run(["git", "clone", "--quiet", REPO_URL, str(repo)], check=True)
os.chdir(repo)
print("Repository ready at", repo)

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
compat_command = ["python", "scripts/kaggle_torch_compat.py"]
if AUTO_FIX_P100:
    compat_command.append("--auto-fix-p100")
subprocess.run(compat_command, check=True)

In [ ]:
runs = repo / "runs"
runs.mkdir(exist_ok=True)
input_root = Path("/kaggle/input")
source_roots = []
if RUNS_INPUT_DIR:
    source_roots.append(Path(RUNS_INPUT_DIR))
archive_paths = [Path(path) for path in RUNS_INPUT_ZIPS]
if RUNS_INPUT_ZIP:
    archive_paths.append(Path(RUNS_INPUT_ZIP))
if AUTO_DISCOVER_RUN_INPUTS and input_root.exists():
    source_roots.append(input_root)
    archive_paths.extend(
        path for path in input_root.rglob("*.zip")
        if path.name in {"runs.zip", "dbzd-runs.zip"}
    )
for archive_index, archive_path in enumerate(dict.fromkeys(archive_paths)):
    if not archive_path.exists():
        raise FileNotFoundError(f"Runs archive not found: {archive_path}")
    restored = work / f"restored-runs-{archive_index}"
    if restored.exists():
        shutil.rmtree(restored)
    shutil.unpack_archive(str(archive_path), restored)
    source_roots.append(restored)
    print("Unpacked prior runs from", archive_path)

candidates = {}
for source_root in dict.fromkeys(path for path in source_roots if path.exists()):
    for summary_path in source_root.rglob("summary.json"):
        try:
            summary = json.loads(summary_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        arm, seed = summary.get("arm"), summary.get("seed")
        revision = (summary.get("config") or {}).get("experiment_revision")
        if revision != EXPERIMENT_REVISION or arm not in {"baseline_matched", "multitask", "dbzd_full", "dbzd_stopgrad"}:
            continue
        if seed is None or summary_path.parent.name != f"{arm}_s{int(seed)}":
            continue
        score = (
            int(bool(summary.get("completed"))),
            sum((summary_path.parent / name).exists() for name in ("model_final.pt", "checkpoint_best.pt", "checkpoint_latest.pt")),
            int((summary_path.parent / "probe_summary.json").exists()),
            int((summary_path.parent / "metrics.csv").exists()),
        )
        key = (str(arm), int(seed))
        if key not in candidates or score > candidates[key][0]:
            candidates[key] = (score, summary_path.parent)
for (arm, seed), (_, source) in sorted(candidates.items()):
    shutil.copytree(source, runs / f"{arm}_s{seed}", dirs_exist_ok=True)
print("Merged revision", EXPERIMENT_REVISION, "runs:", sorted(candidates))

data_dir = repo / "data" / "phase0"
metadata_path = data_dir / "metadata.json"
required_splits = [data_dir / f"{split}.jsonl" for split in ("train", "val", "test")]
data_revision = None
if metadata_path.exists():
    data_revision = json.loads(metadata_path.read_text()).get("generator_schema_version")
data_complete = (
    data_revision == DATAGEN_SCHEMA_VERSION
    and all(path.exists() and path.stat().st_size > 0 for path in required_splits)
)
if not data_complete:
    if data_dir.exists():
        shutil.rmtree(data_dir)
    print("Generating datagen schema", DATAGEN_SCHEMA_VERSION)
    subprocess.run([
        "python", "-m", "datagen", "--output-dir", str(data_dir),
        "--tokenizer", "HuggingFaceTB/SmolLM-135M",
        "--train-n", "40000", "--val-n", "2000", "--test-n", "2000"
    ], check=True)
missing_splits = [str(path) for path in required_splits if not path.exists() or path.stat().st_size == 0]
if missing_splits:
    raise RuntimeError(f"Datagen did not create required splits: {missing_splits}")
print("Dataset ready:", {path.name: path.stat().st_size for path in required_splits})

In [ ]:
import csv, math, yaml
def validate_alpha_run(run_dir):
    summary = json.loads((run_dir / "summary.json").read_text())
    alpha_init = float((summary.get("config") or {}).get("alpha_init", 0.3))
    with (run_dir / "metrics.csv").open() as handle:
        rows = [row for row in csv.DictReader(handle) if row.get("split") == "val"]
    trajectory = [(int(row["global_step"]), float(row["alpha"]), float(row["alpha_lm_gradient"])) for row in rows]
    print("dbzd_full alpha trajectory:", trajectory)
    if not trajectory or any(not math.isfinite(gradient) for _, _, gradient in trajectory):
        raise RuntimeError(f"dbzd_full alpha_lm_gradient is missing/non-finite: {trajectory}")
    if not any(abs(alpha - alpha_init) > 1e-7 for _, alpha, _ in trajectory):
        raise RuntimeError(f"dbzd_full alpha never moved from {alpha_init}: {trajectory}")

def build_export():
    export = work / "dbzd-runs-export"
    if export.exists():
        shutil.rmtree(export)
    export_runs = export / "runs"
    export_runs.mkdir(parents=True)
    artifact_names = {"metrics.csv", "summary.json", "probe_summary.json"}
    for existing_run in sorted(runs.glob("*_s*")):
        if not existing_run.is_dir():
            continue
        target = export_runs / existing_run.name
        target.mkdir()
        for name in artifact_names:
            source = existing_run / name
            if source.exists():
                shutil.copy2(source, target / name)
    if (runs / "analysis").exists():
        shutil.copytree(runs / "analysis", export_runs / "analysis")
    metadata = {
        "title": DATASET_TITLE,
        "id": "local/dbzd-phase0-runs",
        "licenses": [{"name": "CC0-1.0"}],
    }
    (export / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
    return export, export_runs

def publish_results(message):
    export, export_runs = build_export()
    shutil.make_archive(str(work / "dbzd-runs"), "zip", root_dir=export_runs)
    print("Packaged progress:", message, "->", work / "dbzd-runs.zip")

for arm, seed in RUN_MATRIX:
    run_dir = runs / f"{arm}_s{seed}"
    resolved = run_dir / "resolved_config.yaml"
    summary_path = run_dir / "summary.json"
    old_revision = None
    completed_summary = False
    if run_dir.exists():
        old_summary = None
        if summary_path.exists():
            old_summary = json.loads(summary_path.read_text())
            completed_summary = bool(old_summary.get("completed"))
        if resolved.exists():
            old_revision = (yaml.safe_load(resolved.read_text()) or {}).get("experiment_revision")
        elif old_summary is not None:
            old_revision = (old_summary.get("config") or {}).get("experiment_revision")
        if old_revision != EXPERIMENT_REVISION:
            raise RuntimeError(f"Refusing to delete {run_dir}: revision {old_revision!r} != {EXPERIMENT_REVISION!r}")
    if (run_dir / "model_final.pt").exists() or completed_summary:
        print("Skipping completed", arm, seed)
        if RUN_PROBES and not (run_dir / "probe_summary.json").exists():
            if (run_dir / "model_final.pt").exists():
                subprocess.run(["python", "probe.py", "--run-dir", str(run_dir)], check=True)
                if PUBLISH_AFTER_EACH_RUN:
                    publish_results(f"Recovered probe for {arm} seed {seed}")
            else:
                print("WARNING: probe missing and no model checkpoint for", run_dir)
        continue
    command = [
        "python", "train.py", "--config", CONFIG,
        "--data-dir", str(data_dir), "--run-root", str(runs),
        "--arm", arm, "--seed", str(seed)
    ]
    if (run_dir / "checkpoint_latest.pt").exists():
        command.append("--resume")
    print("Running", arm, seed, "resume=" + str("--resume" in command))
    try:
        subprocess.run(command, check=True)
    except subprocess.CalledProcessError:
        publish_results(f"FAILED {arm} seed {seed} in batch {BATCH_ID}")
        raise
    if arm == "dbzd_full":
        try:
            validate_alpha_run(run_dir)
        except Exception:
            publish_results(f"ALPHA CHECK FAILED for {arm} seed {seed}")
            raise
    if PUBLISH_AFTER_EACH_RUN:
        publish_results(f"Training completed for {arm} seed {seed} in batch {BATCH_ID}")
    if RUN_PROBES:
        subprocess.run(["python", "probe.py", "--run-dir", str(run_dir)], check=True)
    if PUBLISH_AFTER_EACH_RUN:
        publish_results(f"Completed {arm} seed {seed} in batch {BATCH_ID}")

if RUN_PROBES:
    checkpoint_names = ("model_final.pt", "checkpoint_best.pt", "checkpoint_latest.pt")
    for summary_path in sorted(runs.glob("*_s*/summary.json")):
        summary = json.loads(summary_path.read_text())
        revision = (summary.get("config") or {}).get("experiment_revision")
        run_dir = summary_path.parent
        if revision != EXPERIMENT_REVISION or not summary.get("completed"):
            continue
        if (run_dir / "probe_summary.json").exists():
            continue
        if (run_dir / "resolved_config.yaml").exists() and any((run_dir / name).exists() for name in checkpoint_names):
            print("Running missing probe for", run_dir.name)
            try:
                subprocess.run(["python", "probe.py", "--run-dir", str(run_dir)], check=True)
            except subprocess.CalledProcessError:
                publish_results(f"PROBE FAILED for {run_dir.name}")
                raise
        else:
            print("WARNING: cannot recover missing probe; checkpoint absent for", run_dir.name)

subprocess.run([
    "python", "analysis.py",
    "--runs-dir", str(runs),
    "--experiment-revision", EXPERIMENT_REVISION,
], check=True)

In [ ]:
publish_results(f"Completed batch {BATCH_ID} with aggregate analysis")
print("Packaged", work / "dbzd-runs.zip")